# Notebook 3: Analyze Visium H&E Data with Squidpy

**Dataset:** Visium H&E stained mouse brain coronal section (pre-processed)

**Reference:** [Squidpy Visium H&E Tutorial](https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_visium_hne.html)

This notebook covers spatial statistics and graph analysis using Squidpy: image feature extraction, neighborhood enrichment, co-occurrence, ligand-receptor interaction, and spatially variable genes (Moran's I).

## 1. Setup & Imports

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import squidpy as sq

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

## 2. Load Pre-processed Data

In [ ]:
img = sq.datasets.visium_hne_image()
adata = sq.datasets.visium_hne_adata()

In [ ]:
sq.pl.spatial_scatter(adata, color="cluster")
plt.savefig("../results/03_visium_hne/01_spatial_clusters.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Image Features

Extract summary features at different scales for multi-scale analysis.

In [ ]:
import matplotlib.pyplot as plt

for scale in [1.0, 2.0]:
    feature_name = f"features_summary_scale{scale}"
    sq.im.calculate_image_features(
        adata,
        img.compute(),
        features="summary",
        key_added=feature_name,
        n_jobs=4,
        scale=scale,
    )

adata.obsm["features"] = pd.concat(
    [adata.obsm[f] for f in adata.obsm.keys() if "features_summary" in f],
    axis="columns",
)
adata.obsm["features"].columns = ad.utils.make_index_unique(
    adata.obsm["features"].columns
)

In [ ]:
def cluster_features(features: pd.DataFrame, like=None) -> pd.Series:
    """Calculate leiden clustering of features."""
    if like is not None:
        features = features.filter(like=like)
    adata = ad.AnnData(features)
    sc.pp.scale(adata)
    sc.pp.pca(adata, n_comps=min(10, features.shape[1] - 1))
    sc.pp.neighbors(adata)
    sc.tl.leiden(adata)
    return adata.obs["leiden"]

adata.obs["features_cluster"] = cluster_features(adata.obsm["features"], like="summary")
sq.pl.spatial_scatter(adata, color=["features_cluster", "cluster"])
plt.savefig("../results/03_visium_hne/01_spatial_clusters.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Spatial Statistics and Graph Analysis

### 4.1 Neighborhood Enrichment

Identify cluster pairs that share a common neighborhood structure.

In [ ]:
sq.gr.spatial_neighbors(adata)
sq.gr.nhood_enrichment(adata, cluster_key="cluster")
sq.pl.nhood_enrichment(adata, cluster_key="cluster")
plt.savefig("../results/03_visium_hne/02_nhood_enrichment.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.2 Co-occurrence Across Spatial Dimensions

Visualize cluster co-occurrence scores across increasing radii.

In [ ]:
sq.gr.co_occurrence(adata, cluster_key="cluster")
sq.pl.co_occurrence(
    adata,
    cluster_key="cluster",
    clusters="Hippocampus",
    figsize=(8, 4),
)
plt.savefig("../results/03_visium_hne/03_co_occurrence.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.3 Ligand-Receptor Interaction Analysis

Run CellPhoneDB-style analysis using Squidpy's ligrec function.

In [ ]:
sq.gr.ligrec(
    adata,
    n_perms=100,
    cluster_key="cluster",
)
sq.pl.ligrec(
    adata,
    cluster_key="cluster",
    source_groups="Hippocampus",
    target_groups=["Pyramidal_layer", "Pyramidal_layer_dentate_gyrus"],
    means_range=(3, np.inf),
    alpha=1e-4,
    swap_axes=True,
)
plt.savefig("../results/03_visium_hne/04_ligrec.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.4 Spatially Variable Genes with Moran's I

Identify genes showing spatial autocorrelation patterns.

In [ ]:
genes = adata[:, adata.var.highly_variable].var_names.values[:1000]
sq.gr.spatial_autocorr(
    adata,
    mode="moran",
    genes=genes,
    n_perms=100,
    n_jobs=1,
)
adata.uns["moranI"].head(10)

In [ ]:
sq.pl.spatial_scatter(adata, color=["Olfm1", "Plp1", "Itpka", "cluster"])
plt.savefig("../results/03_visium_hne/05_moranI_genes.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

- Loaded pre-processed Visium H&E mouse brain data with annotated clusters
- Extracted multi-scale summary image features and clustered them
- Computed neighborhood enrichment → Hippocampus sub-regions are strongly enriched
- Analyzed co-occurrence → Pyramidal_layer co-occurs with Hippocampus at short distances
- Performed ligand-receptor interaction analysis (CellPhoneDB re-implementation)
- Identified spatially variable genes using Moran's I (top: Olfm1, Plp1, Itpka)